In [ ]:
import random
from typing import Iterator

import rdkit
from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem import PandasTools, Draw
from rdkit.Chem.Draw import IPythonConsole

import chython

import pandas as pd

from stereomolgraph import StereoMolGraph

rdkit.__version__

In [ ]:
random.seed(42)
RDLogger.DisableLog('rdApp.*')

Chem.rdmolops.SetUseLegacyStereoPerception = False

IPythonConsole.drawOptions.addStereoAnnotation = True

RENUMBER_TIMES = 20

# Display with molecule rendering and custom image size
PandasTools.RenderImagesInAllDataFrames(images=True)
PandasTools.molRepresentation = 'svg'

In [ ]:
# Read SMILES from text file
def load_smiles_from_txt(filepath: str) -> list[str]:
    """Load SMILES strings from a text file"""
    with open(filepath, 'r') as f:
        smiles_list = [line.strip() for line in f if line.strip()]
    print(f"Loaded {len(smiles_list)} SMILES from {filepath}")
    return smiles_list

# Load SMILES from file
smiles_list = load_smiles_from_txt('../data/inchi_1_06_knownissues_smiles.txt')

In [ ]:
def smiles_to_mol(smiles: str) -> Chem.Mol:
    return Chem.MolFromSmiles(smiles)

def mol_to_inchi(mol: Chem.Mol) -> str:
    molblock = Chem.rdmolfiles.MolToMolBlock(mol)
    return Chem.inchi.MolBlockToInchi(molblock)

def mol_to_canon_smiles(mol: Chem.Mol) -> str:
    return Chem.MolToSmiles(mol, canonical=True)

In [ ]:
def generate_renumbered_molecules(mol: Chem.Mol, n: int = RENUMBER_TIMES
                                  ) -> Iterator[Chem.Mol]:
    """Generate n molecules with renumbered atoms from an rdmol"""
    for _ in range(n):
        # Generate random atom order
        num_atoms = mol.GetNumAtoms()
        new_order = list(range(num_atoms))
        random.shuffle(new_order)
        # Create renumbered molecule
        renumbered_mol = Chem.rdmolops.RenumberAtoms(mol, new_order)
        yield renumbered_mol

In [ ]:
def inchi_identety(mol: Chem.Mol, n: int = RENUMBER_TIMES):
    unique_strings: set[str] = set()
    for renumbered_mol in generate_renumbered_molecules(mol, n):
        unique_strings.add(mol_to_inchi(renumbered_mol))
    if len(unique_strings) == 1:
        return "✓"
    else:
        return "✗"

In [ ]:
def smiles_identety(mol: Chem.Mol, n: int = RENUMBER_TIMES):
    unique_strings: set[str] = set()
    for renumbered_mol in generate_renumbered_molecules(mol, n):
        unique_strings.add(mol_to_canon_smiles(renumbered_mol))
    if len(unique_strings) == 1:
        return "✓"
    else:
        return "✗"

In [ ]:
def smg_identity(mol: Chem.Mol, n: int = RENUMBER_TIMES):
    initial_smg = StereoMolGraph.from_rdmol(mol, stereo_complete=True)
    for renumbered_mol in generate_renumbered_molecules(mol, n):
        renumbered_smg = StereoMolGraph.from_rdmol(renumbered_mol, stereo_complete=True)
        if renumbered_smg != initial_smg:
            return "✗"
    return "✓"

In [ ]:
def chython_identity(mol: Chem.Mol, n: int = RENUMBER_TIMES):
    """Check if chython can identify renumbered molecules as identical"""
    try:
        # Convert initial molecule to chython via SMILES
        initial_smiles = Chem.MolToSmiles(mol)
        initial_chython = chython.smiles(initial_smiles)
        # Check if all renumbered versions are recognized as identical by chython
        for renumbered_mol in generate_renumbered_molecules(mol, n):
            renumbered_smiles = Chem.MolToSmiles(renumbered_mol)
            renumbered_chython = chython.smiles(renumbered_smiles)
            if renumbered_chython != initial_chython:
                return "✗"
        return "✓"
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
def validate_smiles_list(smiles_list: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Validates a list of SMILES strings by checking if they are maintained
    under atom renumbering for InChI, SMILES, SMG, and Chython representations.
    Also checks if each molecule is equal to its enantiomer.
    
    Returns two DataFrames:
    1. Identity checks under renumbering
    """
    identity_results = []
    
    for smiles_input in smiles_list:
        smiles_input, pubchem_cid = smiles_input.split()
        mol = smiles_to_mol(smiles_input)
        if mol is None:
            continue
        
        # Prepare molecule with CIP stereo labels
        mol_with_stereo = mol
        
        identity_results.append({
            'Mol': mol_with_stereo,
            'InChI': inchi_identety(mol),
            'RDKit SMILES': smiles_identety(mol),
            'StereoMolGraph': smg_identity(mol),
            'chython': chython_identity(mol),
            'PubChem_CID': pubchem_cid,
            'Input SMILES': smiles_input
        })
        
    df_identity = pd.DataFrame(identity_results)
    
    return df_identity

In [ ]:
# Validate the smiles list
df_identity = validate_smiles_list(smiles_list)

df_identity.style.set_properties(
    subset=["SMILES"],**{
        "white-space": "pre-wrap",
        "word-wrap": "break-word",
        "max-width": "300px"}
    )

print("Identity Check (Renumbered Self-Comparison):")
display(df_identity)

In [ ]:
# Save both DataFrames as HTML
html_identity = df_identity.to_html(escape=False)
with open('validation_identity.html', 'w') as f:
    f.write(html_identity)
print(f"Saved {len(df_identity)} identity results to validation_identity.html")